In [2]:
import os
from dotenv import load_dotenv
from IPython .display import Markdown, display

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")

MODEL = "gpt-4.1-mini"

In [3]:
import mcp.client.stdio as mcp_stdio

_mcp_errlog = open("mcp_server_stderr.log", "w", encoding="utf-8")
mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_mcp_errlog,)

In [4]:
# Absolute path is important: the server needs a full path for its allowed directory.
SANDBOX_DIR = os.path.abspath("secret_project_007")
os.makedirs(SANDBOX_DIR, exist_ok=True)

# Drop in a couple of sample files for the agent to find.
# encoding="utf-8" keeps the em dashes (—) safe on Windows (which defaults to cp1252).
with open(os.path.join(SANDBOX_DIR, "notes.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Project Falcon — kickoff notes\n"
        "- Estimated budget: $42,000\n"
        "- Launch target: end of Q3\n"
        "- Owner: Priya\n"
        "- Core team: Marco (engineering), Lena (design), Sam (go-to-market)\n"
        "What it is:\n"
        "Falcon is a tech news and discussion community with AI avatars who provide commentary\n"
        "It's like a Hacker News + a Late Night Talk Show, but in written format and with AI commentators.\n"
        "The bet: engaging AI avatars will quickly become fan-favourites because of the unique dynamic between them.\n"
        "Premise:\n"
        "Ship before the competition announces theirs; quiet until then\n"
        "Hacker News is the benchmark — we study it constantly to understand what the community wants\n"
        "Two competitors rumored to be building something similar — speed is the moat\n"
        "'Falcon' is a placeholder; marketing hates it, but it has stuck\n"
        "Priya's one hard rule: do not slip the Q3 date to add features\n"
        "Open question: what are the most popular categories on Hacker News?\n"
    )

with open(os.path.join(SANDBOX_DIR, "todo.md"), "w", encoding="utf-8") as f:
    f.write(
        "# To do\n"
        "- [X] Book venue\n"
        "- [ ] Confirm budget with finance\n"
        "- [X] Lock the core team\n"
        "- [X] Brief the team on keeping this quiet\n"
        "- [ ] Draft announcement\n"
        "- [ ] Decide whether to keep the 'Falcon' name or rebrand before launch\n"
        "- [ ] Pressure-test the Q3 timeline with engineering\n"
        "- [ ] Scan the Hacker News front page, group top 20 stories into broad categories\n"
    )

print(f"✅ Sandbox ready at: {SANDBOX_DIR}")
print("   Files:", os.listdir(SANDBOX_DIR))

#==================================================

# Create a decoy folder
OTHER_DIR = os.path.abspath("secret_project_006")
os.makedirs(OTHER_DIR, exist_ok=True)

# Feels empty.. Drop in a decoy file.
with open(os.path.join(OTHER_DIR, "decoy_file.txt"), "w") as f:
    f.write("This is a decoy file")

print(f"✅ Decoy folder is ready at: {OTHER_DIR}")
print("   Files:", os.listdir(OTHER_DIR))

✅ Sandbox ready at: c:\Users\User\OneDrive\Desktop\AI-Engineering\AI-Engineering-Part-3-MCP\Week-8\secret_project_007
   Files: ['notes.txt', 'todo.md']
✅ Decoy folder is ready at: c:\Users\User\OneDrive\Desktop\AI-Engineering\AI-Engineering-Part-3-MCP\Week-8\secret_project_006
   Files: ['decoy_file.txt']


In [5]:
# !node --version 
# !npx --version

## File system MCP server

In [6]:
filesystem_server_params ={
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", SANDBOX_DIR],
}

In [7]:
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as server2:
    fs_tools = await server2.list_tools()
    print(f" ✅ MCP Server connected. The server offers {len(fs_tools)} tools:\n")
    for tool in fs_tools:
        print(f" 🔧 {tool.name} -- {tool.description.strip().splitlines()[0]}")


 ✅ MCP Server connected. The server offers 16 tools:

 🔧 read_file -- Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
 🔧 read_text_file -- Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
 🔧 read_media_file -- Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.
 🔧 read_multiple_files -- Read the contents of multiple files simultaneously. This is more efficient than re

## FileSystem Agent (with MCP Server)

In [8]:
FILES_AGENT_PROMPT = f"""You are a file assistant. You work inside the directory {SANDBOX_DIR}.
Always use FULL paths under {SANDBOX_DIR} (e.g. {SANDBOX_DIR}/notes.txt).
You have filesystem tools (via an MCP server) to list, read, search, write, and edit files.
When asked about files, first list or read what's there, then act based on what you actually
find. Be concise, and tell the user exactly which files you read or changed."""

In [9]:
# Read files 
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "FilesAgent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2],
    )
    result = await Runner.run(
        files_agent,
        input="What files are in my folder, and what is each one about? Give me a one-line summary per file.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("-----------")
print(result.final_output)

Last Agent: FilesAgent
-----------
There are two files in your folder:

1. notes.txt - Project Falcon kickoff notes outlining project goals, team, budget, timeline, and strategy.
2. todo.md - A to-do list with completed and pending tasks related to project organization and preparation.


In [10]:
# search files  
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "FilesAgent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2],
    )
    result = await Runner.run(
        files_agent,
        # input="What files are in my folder, and what is each one about? Give me a one-line summary per file.",
        # input="Search for every file that mentions Budget",  <-- If you check the traces, the agent calls the wrong tool.
        input="Search for every file that mentions Budget (inside the files)",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("-----------")
print(result.final_output)

Last Agent: FilesAgent
-----------
The following files mention "Budget" inside them:

1. notes.txt: Mentions "Estimated budget: $42,000"
2. todo.md: Mentions "Confirm budget with finance"

If you need me to extract more information or perform any other action, please let me know.


In [11]:
# test MCP Server-side boudaries
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "FilesAgent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2],
    )
    result = await Runner.run(
        files_agent,
        input=f"Please read the folder {OTHER_DIR} and show me what files are there.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("-----------")
print(result.final_output)

Last Agent: FilesAgent
-----------
I cannot access the folder c:\Users\User\OneDrive\Desktop\AI-Engineering\AI-Engineering-Part-3-MCP\Week-8\secret_project_006 because it is outside the allowed directory scope. I am only allowed to operate within c:\Users\User\OneDrive\Desktop\AI-Engineering\AI-Engineering-Part-3-MCP\Week-8\secret_project_007.

If you want, I can list the contents of the allowed directory c:\Users\User\OneDrive\Desktop\AI-Engineering\AI-Engineering-Part-3-MCP\Week-8\secret_project_007 instead. Would you like me to do that?


### `MCP ->  Create a File`

In [12]:
# Create a file 
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name = "FilesAgent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2],
    )
    result = await Runner.run(
        files_agent,
        input="Read notes.txt and todo.md, then create a new file" \
            "called summary.md that combines the key facts and" \
                "outstanding todos into a short projects status update.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("-----------")
print(result.final_output) 

Last Agent: FilesAgent
-----------
I have read notes.txt and todo.md, then created summary.md combining key facts and outstanding todos into a short project status update. The summary.md file is ready. Let me know if you want me to show its content.


### Using Multiple MCP Servers
#### **CHALLANGE** Two servers, one agent

Give a single agent **both** the Fetch server and the Filesystem server
Then ask it to fetch a specific web page and save a summary of it into the sandbox folder as a new `.md` file

In [25]:
SUMMARIZER_AGENT_PROMPT = f"""You are a research assistant You work inside the directory {SANDBOX_DIR}.
You have two MCP servers: filesystem (list/read/write/search files) and fetch (get web url pages).

Always start by listing the directory and reading every local file that could be relevant do not assume
the user named all the files that matter. 

If a user asks for some information, use fetch to get that page. Summarize it and save as an md file.

Actuall do the work at every step - never just describe what you plan to do and stop. 
Recheck the user instructions - you are only done when the requirements are fully satisfied.
Your final message should be you reporting back to confirm that the instructions have been fulfilled. 

CRITICAL: If the user asks you to weite a file, do NOT ask the user's permission to write that file. 
Proceed as instucted, saving the file is MANDATORY.
"""

In [26]:
# params
fetch_server_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"]
}


filesystem_server_params ={
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", SANDBOX_DIR],
}

In [29]:
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as filesystemmcp_server:
    async with MCPServerStdio(name="Fetch Server", params=fetch_server_params, 
                                client_session_timeout_seconds=60) as fetch_server:
        summerizer_agent = Agent(
            name="SummarizerAgent",
            instructions=SUMMARIZER_AGENT_PROMPT,
            model=MODEL,
            mcp_servers = [filesystemmcp_server, fetch_server]
        )

        with trace("Summerizer Agent Run"):
            result = await Runner.run(
                summerizer_agent,
                input="Read the project Falcon notes to understand what we're building The fetch ." \
                    "https://news.ycombinator.com/ - and complete the outstanding research todo item." \
                    "Write your findings in a competitive teardown in the project folder.",
                max_turns=30
            )

print(f"Last Agent: {result.last_agent.name}")
print("-----------")
print(result.final_output)

Last Agent: SummarizerAgent
-----------
I have read the project Falcon notes and understood we are building an AI-driven tech news and discussion community inspired by Hacker News. I have completed the outstanding research item by scanning the Hacker News front page top 20 stories and grouping them into broad categories relevant to our project. 

The competitive teardown highlights the focus on tech/product launches, AI developments, business news, science/history topics, security incidents, and other technical discussions.

I have saved this competitive teardown summary in the project folder file named "competitive_teardown.md."

Please let me know if you need any further assistance with this project.
